# STP Natality — Colab GPU Training

## Before you start (do this once)

**Step 1 — Select GPU runtime**  
`Runtime → Change runtime type → T4 GPU → Save`

**Step 2 — Upload data to Google Drive** (one time only)  
Create this folder path in Drive: `My Drive/stp-natality/data/processed/`  
Upload these 3 files from your laptop:

| File | Size |
|---|---|
| `data/processed/train.parquet` | 181 MB |
| `data/processed/val.parquet` | 64 MB |
| `data/processed/test.parquet` | 111 MB |

After the first upload you never need to upload again.  
All results (MLflow DB, model, figures, log) are automatically saved back to Drive.

---
**Then: `Runtime → Run all`**

In [ ]:
# Cell 1 — Verify GPU
!nvidia-smi

In [ ]:
# Cell 2 — Mount Google Drive (safe to re-run)
import os
if os.path.exists('/content/drive/MyDrive'):
    print('Drive already mounted.')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')

In [ ]:
# Cell 3 — Clone repo + install packages
import os, subprocess, sys

REPO = '/content/stp-natality'
if not os.path.exists(REPO):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Avijeettelkar1/stp-natality.git', REPO],
        check=True
    )
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

os.chdir(REPO)
print('Repo:', os.getcwd())

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lightgbm', 'xgboost', 'optuna', 'mlflow', 'shap',
    'scikit-learn', 'pandas', 'numpy', 'pyarrow', 'matplotlib', 'scipy'
], check=True)
print('Packages installed.')

In [ ]:
# Cell 4 — Verify data files on Drive
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/stp-natality/data/processed')
all_ok = True
for fname in ['train.parquet', 'val.parquet', 'test.parquet']:
    p = DRIVE_DATA / fname
    if p.exists():
        print('  {} : {:.1f} MB  OK'.format(fname, p.stat().st_size / 1e6))
    else:
        print('  {} : NOT FOUND -- upload to Drive first (see instructions above)'.format(fname))
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Upload the 3 parquet files to Drive before continuing.')
print('\nAll data files found. Ready to train.')

In [ ]:
# Cell 5 — Copy data from Drive to local /content (faster reads during training)
import shutil
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/stp-natality/data/processed')
LOCAL_DATA = Path('/content/stp-natality/data/processed')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

for fname in ['train.parquet', 'val.parquet', 'test.parquet']:
    dst = LOCAL_DATA / fname
    if not dst.exists():
        print('Copying {} ...'.format(fname))
        shutil.copy(DRIVE_DATA / fname, dst)
    else:
        print('{} already local.'.format(fname))

print('Data ready.')

In [ ]:
# Cell 6 — Environment check (run this BEFORE training to catch any problems)
import os, sys
from pathlib import Path

print('=== CHECKS ===')
checks = {
    'Drive mounted'  : os.path.exists('/content/drive/MyDrive'),
    'Drive folder'   : os.path.exists('/content/drive/MyDrive/stp-natality'),
    'Repo cloned'    : os.path.exists('/content/stp-natality'),
    'Script exists'  : os.path.exists('/content/stp-natality/run_modeling_colab.py'),
    'train.parquet'  : os.path.exists('/content/stp-natality/data/processed/train.parquet'),
    'val.parquet'    : os.path.exists('/content/stp-natality/data/processed/val.parquet'),
    'test.parquet'   : os.path.exists('/content/stp-natality/data/processed/test.parquet'),
}
for name, ok in checks.items():
    print('{:5s}  {}'.format('OK' if ok else 'FAIL', name))

print('\n=== IMPORTS ===')
sys.path.insert(0, '/content/stp-natality')
for mod in ['src.data.split', 'src.models.pipeline', 'src.models.train', 'src.models.evaluate']:
    try:
        __import__(mod)
        print('OK     {}'.format(mod))
    except Exception as e:
        print('FAIL   {} -- {}'.format(mod, e))

for pkg in ['mlflow', 'lightgbm', 'xgboost', 'optuna', 'shap']:
    try:
        __import__(pkg)
        print('OK     {}'.format(pkg))
    except Exception as e:
        print('FAIL   {} -- {}'.format(pkg, e))

print('\nIf everything shows OK above, run Cell 7 to start training.')

In [ ]:
# Cell 7 — Run full training pipeline (~1-3 hours on T4 GPU)
# Any error will appear as a red traceback here — no more silent failures.
import subprocess, sys, os

os.chdir('/content/stp-natality')

result = subprocess.run(
    [sys.executable, 'run_modeling_colab.py'],
    cwd='/content/stp-natality'
)

print('\nExit code:', result.returncode)
if result.returncode != 0:
    raise RuntimeError('Training failed — read the error printed above.')

In [ ]:
# Cell 8 — Show results and figures
# Self-contained: installs what it needs, reads everything from Drive.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow', 'matplotlib'], check=True)

import mlflow
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/stp-natality')

# Mount Drive if not already mounted
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

mlflow.set_tracking_uri('sqlite:///{}/mlflow.db'.format(DRIVE_ROOT))
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name('stp-natality-phase3')

if exp is None:
    print('No experiment found. Training has not completed yet.')
else:
    all_runs  = client.search_runs(exp.experiment_id, order_by=['start_time DESC'], max_results=100)
    lgbm_runs = [r for r in all_runs if r.data.tags.get('mlflow.runName') == 'lgbm_final']
    xgb_runs  = [r for r in all_runs if r.data.tags.get('mlflow.runName') == 'xgb_final']

    if lgbm_runs:
        m = lgbm_runs[0].data.metrics
        print('=' * 52)
        print('FINAL RESULTS  --  LightGBM (best model)')
        print('=' * 52)
        print('Test MAE         : {:.1f} g   (target < 300 g)'.format(m.get('test_mae', 0)))
        print('Test RMSE        : {:.1f} g'.format(m.get('test_rmse', 0)))
        print('Test R2          : {:.4f}  (target > 0.70)'.format(m.get('test_r2', 0)))
        print('LBW Sensitivity  : {:.3f}  (target > 0.85)'.format(m.get('test_sensitivity', 0)))
        print('LBW Specificity  : {:.3f}'.format(m.get('test_specificity', 0)))
        print('LBW Precision    : {:.3f}'.format(m.get('test_precision', 0)))
        print('LBW F1           : {:.3f}'.format(m.get('test_f1', 0)))
        print('=' * 52)
    else:
        print('lgbm_final run not found — training may still be running.')

    if xgb_runs:
        mx = xgb_runs[0].data.metrics
        print('\nXGBoost val_mae={:.1f}g  val_r2={:.4f}'.format(
            mx.get('val_mae', 0), mx.get('val_r2', 0)))

# Show figures saved to Drive
FIGURES_DIR = DRIVE_ROOT / 'reports' / 'figures'
figs        = ['feature_importance_permutation.png', 'shap_summary.png']
available   = [f for f in figs if (FIGURES_DIR / f).exists()]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(8 * len(available), 6))
    if len(available) == 1:
        axes = [axes]
    for ax, fname in zip(axes, available):
        ax.imshow(mpimg.imread(str(FIGURES_DIR / fname)))
        ax.axis('off')
        ax.set_title(fname.replace('_', ' ').replace('.png', ''), fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No figures found on Drive yet — training may still be running.')